In [1]:
import gc
import numpy as np
import os
import re
import sys
import pandas as pd
import tensorflow_data_validation as tfdv

#VERSION = "20221025"
DATA_DIR = r"Z:\13.Data Science\Projects\2022_GI_Xsell\Data"
PROJECT_DIR = "../"
RAW_DATA_DIR = os.path.join(DATA_DIR, "raw")
PROCESSED_DATA_DIR = os.path.join(DATA_DIR, "processed")
META_DATA_DIR = os.path.join(DATA_DIR, "meta_and_reference_data")
VER_1025_DIR = os.path.join(RAW_DATA_DIR, "20221025")
VER_1118_DIR = os.path.join(RAW_DATA_DIR, "20221118")
VER_1122_DIR = os.path.join(RAW_DATA_DIR, "20221122")
VER_1212_DIR = os.path.join(RAW_DATA_DIR, "20221212")
GI_DATA_DIR = os.path.join(RAW_DATA_DIR, "GI")
PA_DATA_DIR = os.path.join(GI_DATA_DIR, "Travel", "202210")
sys.path.insert(0, DATA_DIR)
sys.path.insert(0, PROJECT_DIR)

from datetime import datetime
from dateutil.relativedelta import relativedelta

# from src.utils.common import PLAN_TYPE_LST, WGP_LST, INFORCE_STATUS_LST
# from src.utils.helper_function import find_plan_type, one_hot

In [2]:
pd.set_option('display.width', None)
pd.set_option('max_colwidth', None)
# pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [3]:
BASESTAT_FILE = os.path.join(PA_DATA_DIR, "basestat.parquet")
CLAMBASE_FILE = os.path.join(PA_DATA_DIR, "clambase_df.parquet")
PREMBASE_FILE = os.path.join(PA_DATA_DIR, "prembase.parquet")
# CLIENT_DATA_MAPPING_FILE = os.path.join(PROCESSED_DATA_DIR, "processed_client_data.parquet")

In [4]:
prembase_df = pd.read_parquet(PREMBASE_FILE)
clambase_df = pd.read_parquet(CLAMBASE_FILE)
basestat_df = pd.read_parquet(BASESTAT_FILE)

: 

: 

In [5]:
prembase_df.head(3)

: 

: 

In [ ]:
clambase_df.head(3)

In [ ]:
basestat_df.head(3)

## import prembase file

In [6]:
prembase_df = pd.read_parquet(PREMBASE_FILE)
# convert column CCDATE, TRANDATE to datetime at prembase_df
prembase_df["OCCDATE"] = pd.to_datetime(prembase_df["OCCDATE"], format="%Y%m%d")
prembase_df["TRANDATE"] = pd.to_datetime(prembase_df["TRANDATE"], format="%Y%m%d")

In [7]:
print("prembase_df shape:", prembase_df.shape)
print("number of policies:", prembase_df['contrnb'].nunique())
print("number of unique clients (using client no.):", prembase_df['COWNNUM'].nunique())

prembase_df shape: (4844657, 79)
number of policies: 2421762
number of unique clients (using client no.): 1499865


In [8]:
prembase_df.columns

Index(['contrnb', 'CCDATE', 'age', 'RSKNO', 'TRANNO', 'DTEEFF', 'gwp',
       'commis', 'disc', 'BATCACTYR', 'BATCACTMN', 'CRATE', 'TRANDATE',
       'trans_code', 'CNTTYPE', 'OCCDATE', 'CRDATE', 'REPNUM', 'COWNNUM',
       'AGNTNUM', 'CNTBRANCH', 'autorenew', 'line', 'ZDMIND', 'ACSART', 'bulk',
       'bulk_name', 'nbrn', 'tranat', 'nwp', 'commis_n', 'ri_inward', 'JACKET',
       'product', 'bulk_name1', 'CLTTYPE', 'CLTSEX', 'CLTDOB', 'MARRYD',
       'NATLTY', 'RSKTYP', 'ldteter', 'risk_des', 'INSURED', 'TOTSIL',
       'ZPLNCD', 'ZPLANCDE', 'trlarea', 'znolive', 'zcmccard_banca', 'plan1',
       'lastcrdate', 'cancdate', 'cancelled', 'dteeff_term', 'duration',
       'lastznolive', 'plan2', 'plan3', 'ChinaCard', 'EnhancedPA',
       'duration_grp', 'NBRN1', 'producer', 'rmisagent', 'rmiscode', 'channel',
       'lifeagent', 'product2', 'gap_trandate_ccdate',
       'gap_trandate_ccdate_gp', 'znolive_gp', 'age_gp', 'loyalty',
       'loyalty_cat', 'insage_smu', 'inssex_smu', 'insage_

In [11]:
BASESTAT_FILE = os.path.join(PA_DATA_DIR, "basestat.parquet")
basestat_df = pd.read_parquet(BASESTAT_FILE)

# Compare Prem vs Base


In [12]:
prembase_column = prembase_df.columns
basestat_column = basestat_df.columns

common_column = basestat_column.intersection(prembase_column)
prem_not_in_base = prembase_column.difference(basestat_column)

print(common_column)
print(prem_not_in_base)

Index(['contrnb', 'CCDATE', 'RSKNO', 'DTEEFF', 'CNTTYPE', 'OCCDATE', 'COWNNUM',
       'AGNTNUM', 'CNTBRANCH', 'autorenew', 'ZDMIND', 'bulk', 'bulk_name',
       'nbrn', 'tranat', 'product', 'bulk_name1', 'CLTSEX', 'INSURED',
       'ZPLANCDE', 'trlarea', 'znolive', 'plan1', 'duration', 'lastznolive',
       'plan2', 'plan3', 'ChinaCard', 'EnhancedPA', 'duration_grp', 'producer',
       'rmisagent', 'channel', 'lifeagent', 'product2', 'gap_trandate_ccdate',
       'gap_trandate_ccdate_gp', 'znolive_gp', 'age_gp', 'loyalty',
       'loyalty_cat', 'insage_smu', 'inssex_smu', 'insage_smu_gp', 'disc_gp',
       'gwp', 'nwp', 'disc', 'commis', 'commis_n'],
      dtype='object')
Index(['ACSART', 'BATCACTMN', 'BATCACTYR', 'CLTDOB', 'CLTTYPE', 'CRATE',
       'CRDATE', 'JACKET', 'MARRYD', 'NATLTY', 'NBRN1', 'REPNUM', 'RSKTYP',
       'TOTSIL', 'TRANDATE', 'TRANNO', 'ZPLNCD', 'age', 'cancdate',
       'cancelled', 'dteeff_term', 'lastcrdate', 'ldteter', 'line',
       'ri_inward', 'risk_des', '

In [47]:
# find number of unique values in each column of prembase_df and basestat_df, output all columns
print (prembase_df['COWNNUM'].nunique())
print(basestat_df['COWNNUM'].nunique())

print (prembase_df['contrnb'].nunique())
print(basestat_df['contrnb'].nunique())



1499865
1432212
2421762
2243681


In [89]:
1499865-1432212

67653

In [49]:
# show some COWNNUM in prembase_df but not in basestat_df
prembase_df[~prembase_df['COWNNUM'].isin(basestat_df['COWNNUM'])]

,contrnb,CCDATE,age,RSKNO,TRANNO,DTEEFF,gwp,commis,disc,BATCACTYR,BATCACTMN,CRATE,TRANDATE,trans_code,CNTTYPE,OCCDATE,CRDATE,REPNUM,COWNNUM,AGNTNUM,CNTBRANCH,autorenew,line,ZDMIND,ACSART,bulk,bulk_name,nbrn,tranat,nwp,commis_n,ri_inward,JACKET,product,bulk_name1,CLTTYPE,CLTSEX,CLTDOB,MARRYD,NATLTY,RSKTYP,ldteter,risk_des,INSURED,TOTSIL,ZPLNCD,ZPLANCDE,trlarea,znolive,zcmccard_banca,plan1,lastcrdate,cancdate,cancelled,dteeff_term,duration,lastznolive,plan2,plan3,ChinaCard,EnhancedPA,duration_grp,NBRN1,producer,rmisagent,rmiscode,channel,lifeagent,product2,gap_trandate_ccdate,gap_trandate_ccdate_gp,znolive_gp,age_gp,loyalty,loyalty_cat,insage_smu,inssex_smu,insage_smu_gp,disc_gp
26089,11005793,20010109.0,39.0,1.0,1.0,20010109.0,675.00,135.00,135.0,2001.0,1.0,1.0,2001-01-04,T405,SMU,2001-01-09,20020108.0,None,0091041,00077,10,0.0,27 - Travel Insurance,N,AXA,N,N,NB,NWBS,0.00000,135.00,N,UOS1100,UNICOVER OVERSEAS STUDENT INS,N,P,M,19610622.0,S,HK,APX,99999999.0,NaN,NaN,1000000.0,None,S2,None,1.0,NaN,NaN,20020108.0,NaN,0.0,99999999.0,365.0,1.0,None,None,N,N,10. >180 Days,NWBS,Direct - Commercial,DIRECT ACCOUNT,DIRC,Agents & Broker,0.0,Other Overseas Student,5.0,2. 1 to 5 days,1. 1 person,3. Age 31 - 40,-0.0,0,NaN,NaN,NaN,3. Discount = 16% to 20%
26090,11005793,20010109.0,39.0,2.0,1.0,20010109.0,1350.00,270.00,270.0,2001.0,1.0,1.0,2001-01-04,T405,SMU,2001-01-09,20020108.0,None,0091041,00077,10,0.0,27 - Travel Insurance,N,AXA,N,N,NB,NWBS,0.00000,270.00,N,UOS1100,UNICOVER OVERSEAS STUDENT INS,N,P,M,19610622.0,S,HK,PBP,99999999.0,NaN,NaN,NaN,None,None,None,NaN,NaN,NaN,20020108.0,NaN,0.0,99999999.0,365.0,NaN,None,None,N,N,10. >180 Days,NWBS,Direct - Commercial,DIRECT ACCOUNT,DIRC,Agents & Broker,0.0,Other Overseas Student,5.0,2. 1 to 5 days,1. 1 person,3. Age 31 - 40,-0.0,0,NaN,NaN,NaN,3. Discount = 16% to 20%
26091,11005793,20010109.0,39.0,3.0,1.0,20010109.0,675.00,135.00,135.0,2001.0,1.0,1.0,2001-01-04,T405,SMU,2001-01-09,20020108.0,None,0091041,00077,10,0.0,27 - Travel Insurance,N,AXA,N,N,NB,NWBS,0.00000,135.00,N,UOS1100,UNICOVER OVERSEAS STUDENT INS,N,P,M,19610622.0,S,HK,HHS,99999999.0,Health,NaN,NaN,None,None,None,NaN,NaN,NaN,20020108.0,NaN,0.0,99999999.0,365.0,NaN,None,None,N,N,10. >180 Days,NWBS,Direct - Commercial,DIRECT ACCOUNT,DIRC,Agents & Broker,0.0,Other Overseas Student,5.0,2. 1 to 5 days,1. 1 person,3. Age 31 - 40,-0.0,0,NaN,NaN,NaN,3. Discount = 16% to 20%
26092,11005796,20001229.0,9999.0,1.0,1.0,20001229.0,675.00,0.00,0.0,2001.0,1.0,1.0,2001-01-08,T405,SMU,2000-12-29,20011228.0,None,0091055,00077,10,0.0,27 - Travel Insurance,N,AXA,N,N,NB,NWBS,0.00000,0.00,N,UOS1100,UNICOVER OVERSEAS STUDENT INS,N,P,M,99999999.0,M,EUR,APX,99999999.0,NaN,NaN,1000000.0,None,S3,None,1.0,NaN,NaN,20011228.0,NaN,0.0,99999999.0,365.0,1.0,None,None,N,N,10. >180 Days,NWBS,Direct - Commercial,DIRECT ACCOUNT,DIRC,Agents & Broker,0.0,Other Overseas Student,-10.0,1. 0 days or les,1. 1 person,9. unknown,-0.0,0,NaN,NaN,NaN,0. No Discount
26093,11005796,20001229.0,9999.0,2.0,1.0,20001229.0,1350.00,0.00,0.0,2001.0,1.0,1.0,2001-01-08,T405,SMU,2000-12-29,20011228.0,None,0091055,00077,10,0.0,27 - Travel Insurance,N,AXA,N,N,NB,NWBS,0.00000,0.00,N,UOS1100,UNICOVER OVERSEAS STUDENT INS,N,P,M,99999999.0,M,EUR,PBP,99999999.0,NaN,NaN,NaN,None,None,None,NaN,NaN,NaN,20011228.0,NaN,0.0,99999999.0,365.0,NaN,None,None,N,N,10. >180 Days,NWBS,Direct - Commercial,DIRECT ACCOUNT,DIRC,Agents & Broker,0.0,Other Overseas Student,-10.0,1. 0 days or les,1. 1 person,9. unknown,-0.0,0,NaN,NaN,NaN,0. No Discount
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4367424,Z0621542,20131228.0,81.0,1.0,1.0,20131228.0,96.25,31.28,0.0,2014.0,1.0,1.0,2013-12-27,T405,STC,2013-12-28,20131229.0,None,0629333,01000,10,0.0,27 - T

In [15]:
# find the shape of prembase_df and basestat_df
print("prembase_df shape:", prembase_df.shape)
print("basestat_df shape:", basestat_df.shape)

prembase_df shape: (4844657, 79)
basestat_df shape: (5086231, 116)


In [51]:
# show earliest OCCDATE of basesat_df
basestat_df['CCDATE'].min()

19990501.0

In [52]:
# show earliest OCCDATE of basesat_df
basestat_df['period'].min()

20140101.0

In [55]:
# show sample of CCDATE and period
basestat_df[['CCDATE', 'period']].sample(10)

,CCDATE,period
968657,20160601.0,20160101.0
554114,20150626.0,20150101.0
4243456,20190829.0,20190101.0
1497942,20160726.0,20160101.0
1335735,20161218.0,20160101.0
3310776,20180108.0,20180101.0
3461732,20180718.0,20190101.0
1337534,20161220.0,20160101.0
3463137,20190901.0,20190101.0
3674674,20190327.0,20190101.0


In [58]:
# show some sample of nbrn and OCCDATE and period of basestat
basestat_df[['contrnb','nbrn', 'CCDATE', 'OCCDATE','period']].sample(10)

,contrnb,nbrn,CCDATE,OCCDATE,period
4591435,66337884,HB,20200103.0,20091103.0,20200101.0
3710622,25698437,NB,20190418.0,20190418.0,20190101.0
5002748,75703088,HB,20220401.0,20011229.0,20220101.0
2148913,25194194,NB,20171019.0,20171019.0,20170101.0
1115285,24776906,NB,20160804.0,20160804.0,20160101.0
2228508,25279079,NB,20171216.0,20171216.0,20170101.0
4119228,85425091,HB,20181005.0,20061005.0,20190101.0
1259755,24855992,NB,20161116.0,20161116.0,20160101.0
586091,S5175853,NB,20150330.0,20150330.0,20150101.0
3510026,25327958,NB,20180209.0,20180209.0,20190101.0


In [63]:
# show rows where contrnb=85425091
basestat_df[['contrnb','RSKNO','nbrn', 'CCDATE', 'OCCDATE','period']][basestat_df['contrnb']=='85425091']

,contrnb,RSKNO,nbrn,CCDATE,OCCDATE,period
534005,85425091,1.0,HB,20151005.0,20061005.0,20150101.0
534006,85425091,2.0,HB,20151005.0,20061005.0,20150101.0
1392408,85425091,1.0,HB,20151005.0,20061005.0,20160101.0
1392409,85425091,1.0,HB,20161005.0,20061005.0,20160101.0
1392410,85425091,2.0,HB,20151005.0,20061005.0,20160101.0
1392411,85425091,2.0,HB,20161005.0,20061005.0,20160101.0
2306813,85425091,1.0,HB,20161005.0,20061005.0,20170101.0
2306814,85425091,1.0,HB,20171005.0,20061005.0,20170101.0
2306815,85425091,2.0,HB,20161005.0,20061005.0,20170101.0
2306816,85425091,2.0,HB,20171005.0,20061005.0,20170101.0


In [67]:
# show rows where contrnb=75703088 and nbrn=NB
basestat_df[['contrnb','RSKNO','nbrn', 'CCDATE', 'OCCDATE','period']][(basestat_df['contrnb']=='66337884') & (basestat_df['nbrn']=='NB')]
# basestat_df[['contrnb','RSKNO','nbrn', 'CCDATE', 'OCCDATE','period']][basestat_df['contrnb']=='75703088']

,contrnb,RSKNO,nbrn,CCDATE,OCCDATE,period


In [78]:
# contrnb =20947813
basestat_df[['contrnb','RSKNO','nbrn', 'CCDATE', 'OCCDATE','period']][basestat_df['contrnb']=='99904261']

,contrnb,RSKNO,nbrn,CCDATE,OCCDATE,period
552574,99904261,1.0,NaN,20150408.0,20150408.0,20150101.0
552575,99904261,2.0,NaN,20150408.0,20150408.0,20150101.0
1415667,99904261,1.0,NaN,20150408.0,20150408.0,20160101.0
1415668,99904261,1.0,HB,20160408.0,20150408.0,20160101.0
1415669,99904261,2.0,NaN,20150408.0,20150408.0,20160101.0
1415670,99904261,2.0,HB,20160408.0,20150408.0,20160101.0
2335327,99904261,1.0,HB,20160408.0,20150408.0,20170101.0
2335328,99904261,1.0,HB,20170408.0,20150408.0,20170101.0
2335329,99904261,2.0,HB,20160408.0,20150408.0,20170101.0
2335330,99904261,2.0,HB,20170408.0,20150408.0,20170101.0


In [72]:
# show rows where nbrn=NB and OCCDATE>20150101
basestat_df[['contrnb','RSKNO','nbrn', 'CCDATE', 'OCCDATE','period']][(basestat_df['nbrn']=='HB') & (basestat_df['OCCDATE']>20150101)]

,contrnb,RSKNO,nbrn,CCDATE,OCCDATE,period
113111,20947813,1.0,HB,20160129.0,20150129.0,20150101.0
113112,20947813,2.0,HB,20160129.0,20150129.0,20150101.0
113204,21606445,1.0,HB,20160323.0,20150323.0,20150101.0
113205,21606445,2.0,HB,20160323.0,20150323.0,20150101.0
527505,66541683,1.0,HB,20151213.0,20150901.0,20150101.0
...,...,...,...,...,...,...
5015168,99896923,1.0,HB,20220406.0,20150406.0,20220101.0
5015169,99896923,2.0,HB,20210406.0,20150406.0,20220101.0
5015170,99896923,2.0,HB,20220406.0,20150406.0,20220101.0
5015171,99904261,1.0,HB,20210408.0,20150408.0,20220101.0


In [17]:
# show some contrnb which it's number of row in prembase_df is different from basestat_df
contrnb_lst = prembase_df['contrnb'].unique()
for i in range(20):
    contrnb=contrnb_lst[i]
    prembase_df_contrnb = prembase_df[prembase_df['contrnb'] == contrnb]
    basestat_df_contrnb = basestat_df[basestat_df['contrnb'] == contrnb]
    if prembase_df_contrnb.shape[0] != basestat_df_contrnb.shape[0]:
        print("contrnb:", contrnb)
        print("prembase_df_contrnb shape:", prembase_df_contrnb.shape)
        print("basestat_df_contrnb shape:", basestat_df_contrnb.shape)

contrnb: 00300028
prembase_df_contrnb shape: (14, 79)
basestat_df_contrnb shape: (20, 116)
contrnb: 00300316
prembase_df_contrnb shape: (22, 79)
basestat_df_contrnb shape: (28, 116)
contrnb: 00300858
prembase_df_contrnb shape: (14, 79)
basestat_df_contrnb shape: (24, 116)
contrnb: 00300963
prembase_df_contrnb shape: (6, 79)
basestat_df_contrnb shape: (9, 116)
contrnb: 00301307
prembase_df_contrnb shape: (10, 79)
basestat_df_contrnb shape: (16, 116)
contrnb: 00301335
prembase_df_contrnb shape: (8, 79)
basestat_df_contrnb shape: (12, 116)
contrnb: 00301996
prembase_df_contrnb shape: (2, 79)
basestat_df_contrnb shape: (3, 116)
contrnb: 00302092
prembase_df_contrnb shape: (6, 79)
basestat_df_contrnb shape: (5, 116)
contrnb: 00302329
prembase_df_contrnb shape: (26, 79)
basestat_df_contrnb shape: (34, 116)
contrnb: 00302566
prembase_df_contrnb shape: (20, 79)
basestat_df_contrnb shape: (22, 116)
contrnb: 00302754
prembase_df_contrnb shape: (8, 79)
basestat_df_contrnb shape: (12, 116)
contrnb

In [19]:
# find contrnb = 00304282 in prembase_df and basestat_df
contrnb = '00304282'
prembase_df_contrnb = prembase_df[prembase_df['contrnb'] == contrnb]
basestat_df_contrnb = basestat_df[basestat_df['contrnb'] == contrnb]



In [81]:
prembase_df_contrnb[['contrnb','RSKNO','nbrn', 'CCDATE', 'OCCDATE']]

,contrnb,RSKNO,nbrn,CCDATE,OCCDATE
204,00304282,1.0,HB,20160119.0,2011-11-16
205,00304282,2.0,HB,20160119.0,2011-11-16


In [88]:
basestat_df_contrnb[['contrnb','RSKNO','nbrn', 'CCDATE', 'OCCDATE','period','nb_clm','paymnt_g','DTEEFF','if_pol']]

,contrnb,RSKNO,nbrn,CCDATE,OCCDATE,period,nb_clm,paymnt_g,DTEEFF,if_pol
691308,00304282,1.0,HB,20160119.0,20111116.0,20160101.0,0.0,0.0,20160119.0,1.0
691309,00304282,2.0,HB,20160119.0,20111116.0,20160101.0,0.0,0.0,20160119.0,0.0
1606242,00304282,1.0,HB,20160119.0,20111116.0,20170101.0,0.0,0.0,20160119.0,0.0
1606243,00304282,2.0,HB,20160119.0,20111116.0,20170101.0,0.0,0.0,20160119.0,0.0


In [36]:
prembase_df_contrnb[common_column]

,contrnb,CCDATE,RSKNO,DTEEFF,CNTTYPE,OCCDATE,COWNNUM,AGNTNUM,CNTBRANCH,autorenew,ZDMIND,bulk,bulk_name,nbrn,tranat,product,bulk_name1,CLTSEX,INSURED,ZPLANCDE,trlarea,znolive,plan1,duration,lastznolive,plan2,plan3,ChinaCard,EnhancedPA,duration_grp,producer,rmisagent,channel,lifeagent,product2,gap_trandate_ccdate,gap_trandate_ccdate_gp,znolive_gp,age_gp,loyalty,loyalty_cat,insage_smu,inssex_smu,insage_smu_gp,disc_gp,gwp,nwp,disc,commis,commis_n
204,00304282,20160119.0,1.0,20160119.0,STM,2011-11-16,O7938329,E05296,20,0.0,N,N,N,HB,RNWL,BUSINESS TRAVELCARE,N,NaN,NaN,11,1,1.0,NaN,366.0,1.0,Area 1,1 Person,N,NA,10. >180 Days,HSBC - Dummy,COB-BIBONLINE,Banca - HSBC,0.0,Banca Group Travel,-16.0,1. 0 days or les,1. 1 person,9. unknown,4.0,4,NaN,NaN,NaN,0. No Discount,445.8,440.2275,0.0,89.16,89.16
205,00304282,20160119.0,2.0,20160119.0,STM,2011-11-16,O7938329,E05296,20,0.0,N,N,N,HB,RNWL,BUSINESS TRAVELCARE,N,NaN,NaN,11,1,NaN,NaN,0.0,NaN,Area 1,1 Person,N,NA,10. >180 Days,HSBC - Dummy,COB-BIBONLINE,Banca - HSBC,0.0,Banca Group Travel,-16.0,1. 0 days or les,1. 1 person,9. unknown,4.0,4,NaN,NaN,NaN,0. No Discount,826.2,815.8725,0.0,165.24,165.24


In [35]:
# only show columns in common between prembase_df and basestat_df
basestat_df_contrnb[common_column]

,contrnb,CCDATE,RSKNO,DTEEFF,CNTTYPE,OCCDATE,COWNNUM,AGNTNUM,CNTBRANCH,autorenew,ZDMIND,bulk,bulk_name,nbrn,tranat,product,bulk_name1,CLTSEX,INSURED,ZPLANCDE,trlarea,znolive,plan1,duration,lastznolive,plan2,plan3,ChinaCard,EnhancedPA,duration_grp,producer,rmisagent,channel,lifeagent,product2,gap_trandate_ccdate,gap_trandate_ccdate_gp,znolive_gp,age_gp,loyalty,loyalty_cat,insage_smu,inssex_smu,insage_smu_gp,disc_gp,gwp,nwp,disc,commis,commis_n
691308,00304282,20160119.0,1.0,20160119.0,STM,20111116.0,O7938329,E05296,20,0.0,N,N,N,HB,RNWL,BUSINESS TRAVELCARE,N,NaN,NaN,11,1,1.0,NaN,366.0,1.0,Area 1,1 Person,N,NA,10. >180 Days,HSBC - Dummy,COB-BIBONLINE,Banca - HSBC,0.0,Banca Group Travel,-16.0,1. 0 days or les,1. 1 person,9. unknown,4.0,4,NaN,NaN,NaN,0. No Discount,445.8,440.2275,0.0,89.16,89.16
691309,00304282,20160119.0,2.0,20160119.0,STM,20111116.0,O7938329,E05296,20,0.0,N,N,N,HB,RNWL,BUSINESS TRAVELCARE,N,NaN,NaN,11,1,NaN,NaN,0.0,NaN,Area 1,1 Person,N,NA,10. >180 Days,HSBC - Dummy,COB-BIBONLINE,Banca - HSBC,0.0,Banca Group Travel,-16.0,1. 0 days or les,1. 1 person,9. unknown,4.0,4,NaN,NaN,NaN,0. No Discount,826.2,815.8725,0.0,165.24,165.24
1606242,00304282,20160119.0,1.0,20160119.0,STM,20111116.0,O7938329,E05296,20,0.0,N,N,N,HB,RNWL,BUSINESS TRAVELCARE,N,NaN,NaN,11,1,1.0,NaN,366.0,1.0,Area 1,1 Person,N,NA,10. >180 Days,HSBC - Dummy,COB-BIBONLINE,Banca - HSBC,0.0,Banca Group Travel,-16.0,1. 0 days or les,1. 1 person,9. unknown,4.0,4,NaN,NaN,NaN,0. No Discount,0.0,0.0000,0.0,0.00,0.00
1606243,00304282,20160119.0,2.0,20160119.0,STM,20111116.0,O7938329,E05296,20,0.0,N,N,N,HB,RNWL,BUSINESS TRAVELCARE,N,NaN,NaN,11,1,NaN,NaN,0.0,NaN,Area 1,1 Person,N,NA,10. >180 Days,HSBC - Dummy,COB-BIBONLINE,Banca - HSBC,0.0,Banca Group Travel,-16.0,1. 0 days or les,1. 1 person,9. unknown,4.0,4,NaN,NaN,NaN,0. No Discount,0.0,0.0000,0.0,0.00,0.00


In [37]:
# prembase_df_contrnb drop duplicate rows
prembase_df_contrnb.drop_duplicates()
prembase_df_contrnb

,contrnb,CCDATE,age,RSKNO,TRANNO,DTEEFF,gwp,commis,disc,BATCACTYR,BATCACTMN,CRATE,TRANDATE,trans_code,CNTTYPE,OCCDATE,CRDATE,REPNUM,COWNNUM,AGNTNUM,CNTBRANCH,autorenew,line,ZDMIND,ACSART,bulk,bulk_name,nbrn,tranat,nwp,commis_n,ri_inward,JACKET,product,bulk_name1,CLTTYPE,CLTSEX,CLTDOB,MARRYD,NATLTY,RSKTYP,ldteter,risk_des,INSURED,TOTSIL,ZPLNCD,ZPLANCDE,trlarea,znolive,zcmccard_banca,plan1,lastcrdate,cancdate,cancelled,dteeff_term,duration,lastznolive,plan2,plan3,ChinaCard,EnhancedPA,duration_grp,NBRN1,producer,rmisagent,rmiscode,channel,lifeagent,product2,gap_trandate_ccdate,gap_trandate_ccdate_gp,znolive_gp,age_gp,loyalty,loyalty_cat,insage_smu,inssex_smu,insage_smu_gp,disc_gp
204,00304282,20160119.0,9999.0,1.0,2.0,20160119.0,445.8,89.16,0.0,2016.0,2.0,1.0,2016-02-04,T413,STM,2011-11-16,20170118.0,None,O7938329,E05296,20,0.0,27 - Travel Insurance,N,AXA,N,N,HB,RNWL,440.2275,89.16,N,BSTM0116,BUSINESS TRAVELCARE,N,C,NaN,NaN,NaN,None,ATH,99999999.0,PA,NaN,600000.0,1,11,1,1.0,N,NaN,20170118.0,NaN,0.0,99999999.0,366.0,1.0,Area 1,1 Person,N,NA,10. >180 Days,RNWL,HSBC - Dummy,COB-BIBONLINE,9998,Banca - HSBC,0.0,Banca Group Travel,-16.0,1. 0 days or les,1. 1 person,9. unknown,4.0,4,NaN,NaN,NaN,0. No Discount
205,00304282,20160119.0,9999.0,2.0,2.0,20160119.0,826.2,165.24,0.0,2016.0,2.0,1.0,2016-02-04,T413,STM,2011-11-16,20170118.0,None,O7938329,E05296,20,0.0,27 - Travel Insurance,N,AXA,N,N,HB,RNWL,815.8725,165.24,N,BSTM0116,BUSINESS TRAVELCARE,N,C,NaN,NaN,NaN,None,ATM,99999999.0,Miscellaneous,NaN,600000.0,1,11,1,NaN,N,NaN,20170118.0,NaN,0.0,99999999.0,0.0,NaN,Area 1,1 Person,N,NA,10. >180 Days,RNWL,HSBC - Dummy,COB-BIBONLINE,9998,Banca - HSBC,0.0,Banca Group Travel,-16.0,1. 0 days or les,1. 1 person,9. unknown,4.0,4,NaN,NaN,NaN,0. No Discount


In [40]:
# basestat_df_contrnb drop duplicate rows
basestat_df_contrnb.drop_duplicates()

,contrnb,CCDATE,RSKNO,DTEEFF,CNTTYPE,OCCDATE,COWNNUM,AGNTNUM,CNTBRANCH,autorenew,ZDMIND,bulk,bulk_name,nbrn,tranat,product,bulk_name1,CLTSEX,INSURED,ZPLANCDE,trlarea,znolive,plan1,duration,lastznolive,plan2,plan3,ChinaCard,EnhancedPA,duration_grp,producer,rmisagent,channel,lifeagent,product2,gap_trandate_ccdate,gap_trandate_ccdate_gp,znolive_gp,age_gp,loyalty,loyalty_cat,insage_smu,inssex_smu,insage_smu_gp,disc_gp,gwp,nwp,gep,nep,yduration,disc,ypol,ypol_actual,no_pol,no_rsk,commis,commis_n,nec,if_pol,if_rsk,ExpectedCost,TechnicalPremium,MarketingPremium,GWP_initial,APTP_gp,chgbal_g,chgbal_n,paymnt_g,paymnt_n,nb_clm,incur_g,incur_n,ultincur_n,ult_clm,lrg_incur_g,lrg_incur_n,lrg_ultincur,if_znolive,Agg_CA_nb_clm,Agg_EA_nb_clm,Agg_ME_nb_clm,Agg_PA_nb_clm,Agg_PB_nb_clm,Agg_TD_nb_clm,Agg_TP_nb_clm,Agg_CA_ultclm,Agg_EA_ultclm,Agg_ME_ultclm,Agg_PA_ultclm,Agg_PB_ultclm,Agg_TD_ultclm,Agg_TP_ultclm,Agg_CA_incur_g,Agg_EA_incur_g,Agg_ME_incur_g,Agg_PA_incur_g,Agg_PB_incur_g,Agg_TD_incur_g,Agg_TP_incur_g,Agg_CA_incur_n,Agg_EA_incur_n,Agg_ME_incur_n,Agg_PA_incur_n,Agg_PB_incur_n,Agg_TD_incur_n,Agg_TP_incur_n,Agg_CA_ultincur,Agg_EA_ultincur,Agg_ME_ultincur,Agg_PA_ultincur,Agg_PB_ultincur,Agg_TD_ultincur,Agg_TP_ultincur,period,total_yduration,no_znolive
691308,00304282,20160119.0,1.0,20160119.0,STM,20111116.0,O7938329,E05296,20,0.0,N,N,N,HB,RNWL,BUSINESS TRAVELCARE,N,NaN,NaN,11,1,1.0,NaN,366.0,1.0,Area 1,1 Person,N,NA,10. >180 Days,HSBC - Dummy,COB-BIBONLINE,Banca - HSBC,0.0,Banca Group Travel,-16.0,1. 0 days or les,1. 1 person,9. unknown,4.0,4,NaN,NaN,NaN,0. No Discount,445.8,440.2275,423.875410,418.576967,348.0,0.0,0.952772,0.95082,1.0,1.0,89.16,89.16,84.775082,1.0,1.0,0.0,0.0,0.0,445.8,12. unknown,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,20160101.0,348.0,1.0
691309,00304282,20160119.0,2.0,20160119.0,STM,20111116.0,O7938329,E05296,20,0.0,N,N,N,HB,RNWL,BUSINESS TRAVELCARE,N,NaN,NaN,11,1,NaN,NaN,0.0,NaN,Area 1,1 Person,N,NA,10. >180 Days,HSBC - Dummy,COB-BIBONLINE,Banca - HSBC,0.0,Banca Group Travel,-16.0,1. 0 days or les,1. 1 person,9. unknown,4.0,4,NaN,NaN,NaN,0. No Discount,826.2,815.8725,785.567213,775.747623,348.0,0.0,0.000000,0.00000,0.0,1.0,165.24,165.24,157.113443,0.0,1.0,0.0,0.0,0.0,826.2,12. unknown,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,20160101.0,0.0,0.0
1606242,00304282,20160119.0,1.0,20160119.0,STM,20111116.0,O7938329,E05296,20,0.0,N,N,N,HB,RNWL,BUSINESS TRAVELCARE,N,NaN,NaN,11,1,1.0,NaN,366.0,1.0,Area 1,1 Person,N,NA,10. >180 Days,HSBC - Dummy,COB-BIBONLINE,Banca - HSBC,0.0,Banca Group Travel,-16.0,1. 0 days or les,1. 1 person,9. unknown,4.0,4,NaN,NaN,NaN,0. No Discount,0.0,0.0000,21.924590,21.650533,18.0,0.0,0.049281,0.00000,0.0,0.0,0.00,0.00,4.384918,0.0,0.0,0.0,0.0,0.0,0.0,12. unknown,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,20170101.0,18.0,0.0
1606243,00304282,20160119.0,2.0,20160119.0,STM,20111116.0,O7938329,E05296,20,0.0,N,N,N,HB,RNWL,BUSINESS TRAVELCARE,N,NaN,NaN,11,1,NaN,NaN,0.0,NaN,Area 1,1 Person,N,NA,10. >180 Days,HSBC - Dummy,COB-BIBONLINE,Banca - HSBC,0.0,Banca Group Travel,-16.0,1. 0 days or les,1. 1 person,9. unknown,4.0,4,NaN,NaN,NaN,0. No Discount,0.0,0.0000,40.632787,40.124877,18.0,0.0,0.000000,0.00000,0.0,0.0,0.00,0.00,8.126557,0.0,0.0,0.0,0.0,0.0,0.0,12. unknown,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,20170101.0,0.0,0.0


In [41]:
# show col=period in both df
prembase_df_contrnb['period']


KeyError: 'period'

In [42]:
basestat_df_contrnb['period']

691308     20160101.0
691309     20160101.0
1606242    20170101.0
1606243    20170101.0
Name: period, dtype: float64

## import clambase file

In [ ]:
clambase_df = pd.read_parquet(CLAMBASE_FILE)

# # convert column CCDATE, TRANDATE to datetime at basestat_df
# basestat_df["OCCDATE"] = pd.to_datetime(basestat_df["OCCDATE"], format="%Y%m%d")
# basestat_df["TRANDATE"] = pd.to_datetime(basestat_df["TRANDATE"], format="%Y%m%d")

In [ ]:
print("clambase_df shape:", clambase_df.shape)
print("number of policies:", clambase_df['contrnb'].nunique())
# print("number of unique clients (using client no.):", clambase_df['COWNNUM'].nunique())

clambase_df shape: (227202, 364)
number of policies: 138324


In [ ]:
clambase_df.head()

,CLAIM,CNTTYPE,RSKNO,DATOCC,DATREP,CLSTAT,MEVENT,SERVBR,contrnb,ldf,clmldf,zdmind_clm,clmdsc,RSCD,rscd_des,chgbal_n,paymnt_n,chgbal_g,paymnt_g,incur_g,incur_n,ultincur_n,ay,am,nb_clm,ult_clm,nb_rscd,large,lrg_nb_clm,AF_nb_clm,AF_ultclm,AF_incur_g,AF_incur_n,AF_ultincur,AL_nb_clm,AL_ultclm,AL_incur_g,AL_incur_n,AL_ultincur,AQ_nb_clm,AQ_ultclm,AQ_incur_g,AQ_incur_n,AQ_ultincur,AW_nb_clm,AW_ultclm,AW_incur_g,AW_incur_n,AW_ultincur,BD_nb_clm,BD_ultclm,BD_incur_g,BD_incur_n,BD_ultincur,BP_nb_clm,BP_ultclm,BP_incur_g,BP_incur_n,BP_ultincur,BU_nb_clm,BU_ultclm,BU_incur_g,BU_incur_n,BU_ultincur,CA_nb_clm,CA_ultclm,CA_incur_g,CA_incur_n,CA_ultincur,CB_nb_clm,CB_ultclm,CB_incur_g,CB_incur_n,CB_ultincur,CD_nb_clm,CD_ultclm,CD_incur_g,CD_incur_n,CD_ultincur,CE_nb_clm,CE_ultclm,CE_incur_g,CE_incur_n,CE_ultincur,CI_nb_clm,CI_ultclm,CI_incur_g,CI_incur_n,CI_ultincur,CK_nb_clm,CK_ultclm,CK_incur_g,CK_incur_n,CK_ultincur,CM_nb_clm,CM_ultclm,CM_incur_g,CM_incur_n,CM_ultincur,CO_nb_clm,CO_ultclm,CO_incur_g,CO_incur_n,CO_ultincur,CS_nb_clm,CS_ultclm,CS_incur_g,CS_incur_n,CS_ultincur,CU_nb_clm,CU_ultclm,CU_incur_g,CU_incur_n,CU_ultincur,DE_nb_clm,DE_ultclm,DE_incur_g,DE_incur_n,DE_ultincur,DI_nb_clm,DI_ultclm,DI_incur_g,DI_incur_n,DI_ultincur,DN_nb_clm,DN_ultclm,DN_incur_g,DN_incur_n,DN_ultincur,DP_nb_clm,DP_ultclm,DP_incur_g,DP_incur_n,DP_ultincur,DY_nb_clm,DY_ultclm,DY_incur_g,DY_incur_n,DY_ultincur,EA_nb_clm,EA_ultclm,EA_incur_g,EA_incur_n,EA_ultincur,EF_nb_clm,EF_ultclm,EF_incur_g,EF_incur_n,EF_ultincur,EV_nb_clm,EV_ultclm,EV_incur_g,EV_incur_n,EV_ultincur,FE_nb_clm,FE_ultclm,FE_incur_g,FE_incur_n,FE_ultincur,FG_nb_clm,FG_ultclm,FG_incur_g,FG_incur_n,FG_ultincur,GH_nb_clm,GH_ultclm,GH_incur_g,GH_incur_n,GH_ultincur,GQ_nb_clm,GQ_ultclm,GQ_incur_g,GQ_incur_n,GQ_ultincur,HC_nb_clm,HC_ultclm,HC_incur_g,HC_incur_n,HC_ultincur,HE_nb_clm,HE_ultclm,HE_incur_g,HE_incur_n,HE_ultincur,HS_nb_clm,HS_ultclm,HS_incur_g,HS_incur_n,HS_ultincur,IC_nb_clm,IC_ultclm,IC_incur_g,IC_incur_n,IC_ultincur,IF_nb_clm,IF_ultclm,IF_incur_g,IF_incur_n,IF_ultincur,IN_nb_clm,IN_ultclm,IN_incur_g,IN_incur_n,IN_ultincur,IP_nb_clm,IP_ultclm,IP_incur_g,IP_incur_n,IP_ultincur,LK_nb_clm,LK_ultclm,LK_incur_g,LK_incur_n,LK_ultincur,ME_nb_clm,ME_ultclm,ME_incur_g,ME_incur_n,ME_ultincur,MO_nb_clm,MO_ultclm,MO_incur_g,MO_incur_n,MO_ultincur,MS_nb_clm,MS_ultclm,MS_incur_g,MS_incur_n,MS_ultincur,OD_nb_clm,OD_ultclm,OD_incur_g,OD_incur_n,OD_ultincur,OL_nb_clm,OL_ultclm,OL_incur_g,OL_incur_n,OL_ultincur,OP_nb_clm,OP_ultclm,OP_incur_g,OP_incur_n,OP_ultincur,OT_nb_clm,OT_ultclm,OT_incur_g,OT_incur_n,OT_ultincur,PA_nb_clm,PA_ultclm,PA_incur_g,PA_incur_n,PA_ultincur,PB_nb_clm,PB_ultclm,PB_incur_g,PB_incur_n,PB_ultincur,PE_nb_clm,PE_ultclm,PE_incur_g,PE_incur_n,PE_ultincur,PL_nb_clm,PL_ultclm,PL_incur_g,PL_incur_n,PL_ultincur,PO_nb_clm,PO_ultclm,PO_incur_g,PO_incur_n,PO_ultincur,PQ_nb_clm,PQ_ultclm,PQ_incur_g,PQ_incur_n,PQ_ultincur,PW_nb_clm,PW_ultclm,PW_incur_g,PW_incur_n,PW_ultincur,RC_nb_clm,RC_ultclm,RC_incur_g,RC_incur_n,RC_ultincur,RE_nb_clm,RE_ultclm,RE_incur_g,RE_incur_n,RE_ultincur,RH_nb_clm,RH_ultclm,RH_incur_g,RH_incur_n,RH_ultincur,SA_nb_clm,SA_ultclm,SA_incur_g,SA_incur_n,SA_ultincur,SI_nb_clm,SI_ultclm,SI_incur_g,SI_incur_n,SI_ultincur,SK_nb_clm,SK_ultclm,SK_incur_g,SK_incur_n,SK_ultincur,TB_nb_clm,TB_ultclm,TB_incur_g,TB_incur_n,TB_ultincur,TC_nb_clm,TC_ultclm,TC_incur_g,TC_incur_n,TC_ultincur,TD_nb_clm,TD_ultclm,TD_incur_g,TD_incur_n,TD_ultincur,TH_nb_clm,TH_ultclm,TH_incur_g,TH_incur_n,TH_ultincur,TI_nb_clm,TI_ultclm,TI_incur_g,TI_incur_n,TI_ultincur,TP_nb_clm,TP_ultclm,TP_incur_g,TP_incur_n,TP_ultincur,UI_nb_clm,UI_ultclm,UI_incur_g,UI_incur_n,UI_ultincur,WB_nb_clm,WB_ultclm,WB_incur_g,WB_incur_n,WB_ultincur,lrg_incur_g,lrg_incur_n,lrg_ultincur,ClosedRSCD,ClosedClm,PLASIC,COUNTRY,DTEEFF,CCDATE,ZDMIND
0,Y0383836,STX,2.0,20160228.0,20160314.0,2,NA,20,00300316,0.952674,1.0,N,Damage to baggage ella,PB,PERSONAL EFFECTS,0.0,650.0,0.0,650.0,650.0,650.0,619.23810,2016.0,201602.0,1.0,1.

## Compare DFs

In [ ]:
prembase_column = prembase_df.columns
clambase_column = clambase_df.columns

common_column = prembase_column.intersection(clambase_column)
prem_not_in_clam = prembase_column.difference(clambase_column)

print(common_column)
print(prem_not_in_clam)

Index(['contrnb', 'CCDATE', 'RSKNO', 'DTEEFF', 'CNTTYPE', 'ZDMIND'], dtype='object')
Index(['ACSART', 'AGNTNUM', 'BATCACTMN', 'BATCACTYR', 'CLTDOB', 'CLTSEX',
       'CLTTYPE', 'CNTBRANCH', 'COWNNUM', 'CRATE', 'CRDATE', 'ChinaCard',
       'EnhancedPA', 'INSURED', 'JACKET', 'MARRYD', 'NATLTY', 'NBRN1',
       'OCCDATE', 'REPNUM', 'RSKTYP', 'TOTSIL', 'TRANDATE', 'TRANNO',
       'ZPLANCDE', 'ZPLNCD', 'age', 'age_gp', 'autorenew', 'bulk', 'bulk_name',
       'bulk_name1', 'cancdate', 'cancelled', 'channel', 'commis', 'commis_n',
       'disc', 'disc_gp', 'dteeff_term', 'duration', 'duration_grp',
       'gap_trandate_ccdate', 'gap_trandate_ccdate_gp', 'gwp', 'insage_smu',
       'insage_smu_gp', 'inssex_smu', 'lastcrdate', 'lastznolive', 'ldteter',
       'lifeagent', 'line', 'loyalty', 'loyalty_cat', 'nbrn', 'nwp', 'plan1',
       'plan2', 'plan3', 'producer', 'product', 'product2', 'ri_inward',
       'risk_des', 'rmisagent', 'rmiscode', 'tranat', 'trans_code', 'trlarea',
       'zcmcc

In [2]:
prembase_column = prembase_df.columns
clambase_column = clambase_df.columns

common_column = prembase_column.intersection(clambase_column)
clam_not_in_prem = clambase_column.difference(prembase_column)

print(common_column)
print(list(clam_not_in_prem))


NameError: name 'prembase_df' is not defined

In [17]:
common_col=['contrnb', 'CCDATE', 'DTEEFF']

In [18]:
p_df=prembase_df[['COWNNUM', 'contrnb', 'CCDATE','DTEEFF']].copy()

In [25]:
c_df=clambase_df[['paymnt_g', 'contrnb', 'CCDATE','DTEEFF']].copy()

In [20]:
merged=pd.merge(p_df, c_df, on=common_col, how='outer')

In [21]:
merged.shape

(4957448, 5)

In [22]:
BASESTAT_FILE = os.path.join(PA_DATA_DIR, "basestat.parquet")
basestat_df = pd.read_parquet(BASESTAT_FILE)

In [26]:
b_df=basestat_df[['COWNNUM', 'contrnb', 'CCDATE','DTEEFF',paymnt_g]]

NameError: name 'paymnt_g' is not defined

In [27]:
# prembase_column = prembase_df.columns
clambase_column = clambase_df.columns
basestat_column = basestat_df.columns

common_column = basestat_column.intersection(clambase_column)
clam_not_in_base = clambase_column.difference(basestat_column)

print(common_column)
print(clam_not_in_base)

Index(['contrnb', 'CCDATE', 'RSKNO', 'DTEEFF', 'CNTTYPE', 'ZDMIND', 'chgbal_g',
       'chgbal_n', 'paymnt_g', 'paymnt_n', 'nb_clm', 'incur_g', 'incur_n',
       'ultincur_n', 'ult_clm', 'lrg_incur_g', 'lrg_incur_n', 'lrg_ultincur'],
      dtype='object')
Index(['AF_incur_g', 'AF_incur_n', 'AF_nb_clm', 'AF_ultclm', 'AF_ultincur',
       'AL_incur_g', 'AL_incur_n', 'AL_nb_clm', 'AL_ultclm', 'AL_ultincur',
       ...
       'am', 'ay', 'clmdsc', 'clmldf', 'large', 'ldf', 'lrg_nb_clm', 'nb_rscd',
       'rscd_des', 'zdmind_clm'],
      dtype='object', length=346)


In [28]:
prembase_column = prembase_df.columns
clambase_column = clambase_df.columns
basestat_column = basestat_df.columns

common_column = basestat_column.intersection(prembase_column)
prem_not_in_base = prembase_column.difference(basestat_column)

print(common_column)
print(prem_not_in_base)

Index(['contrnb', 'CCDATE', 'RSKNO', 'DTEEFF', 'CNTTYPE', 'OCCDATE', 'COWNNUM',
       'AGNTNUM', 'CNTBRANCH', 'autorenew', 'ZDMIND', 'bulk', 'bulk_name',
       'nbrn', 'tranat', 'product', 'bulk_name1', 'CLTSEX', 'INSURED',
       'ZPLANCDE', 'trlarea', 'znolive', 'plan1', 'duration', 'lastznolive',
       'plan2', 'plan3', 'ChinaCard', 'EnhancedPA', 'duration_grp', 'producer',
       'rmisagent', 'channel', 'lifeagent', 'product2', 'gap_trandate_ccdate',
       'gap_trandate_ccdate_gp', 'znolive_gp', 'age_gp', 'loyalty',
       'loyalty_cat', 'insage_smu', 'inssex_smu', 'insage_smu_gp', 'disc_gp',
       'gwp', 'nwp', 'disc', 'commis', 'commis_n'],
      dtype='object')
Index(['ACSART', 'BATCACTMN', 'BATCACTYR', 'CLTDOB', 'CLTTYPE', 'CRATE',
       'CRDATE', 'JACKET', 'MARRYD', 'NATLTY', 'NBRN1', 'REPNUM', 'RSKTYP',
       'TOTSIL', 'TRANDATE', 'TRANNO', 'ZPLNCD', 'age', 'cancdate',
       'cancelled', 'dteeff_term', 'lastcrdate', 'ldteter', 'line',
       'ri_inward', 'risk_des', '

## Merge two DFs

In [14]:
merged=pd.merge(prembase_df, clambase_df, on=common_col, how='outer')

MemoryError: Unable to allocate 12.7 GiB for an array with shape (344, 4936307) and data type float64

print("merged shape:", merged.shape)
print("number of policies:", merged['contrnb'].nunique())
print("number of unique clients (using client no.):", merged['COWNNUM'].nunique())